In [2]:
# Importar bibliotecas necessárias
import os
from dotenv import load_dotenv
import kaggle
import pandas as pd
from minio import Minio
from minio.error import S3Error
import zipfile

# Carregar variáveis de ambiente do .env
load_dotenv()

# Configurações do Kaggle
os.environ['KAGGLE_USERNAME'] = os.getenv('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = os.getenv('KAGGLE_KEY')

# Configurações do MinIO
MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')

print("Configurações carregadas com sucesso!")

# Baixar dataset do Kaggle
try:
    kaggle.api.dataset_download_files('utkarshx27/non-alcohol-fatty-liver-disease', path='../data/raw', unzip=False)
    print("Download do dataset concluído com sucesso!")
except kaggle.rest.ApiException as e:
    print(f"Erro ao baixar dataset: {e}")

# Descompactar os arquivos do dataset
try:
    with zipfile.ZipFile('../data/raw/non-alcohol-fatty-liver-disease.zip', 'r') as zip_ref:
        zip_ref.extractall('../data/raw')
    print("Descompactação concluída com sucesso!")
except zipfile.BadZipFile as e:
    print(f"Erro ao descompactar o arquivo: {e}")

# Configurar o cliente MinIO
client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False
)

# Nome dos buckets
buckets = ["raw", "bronze", "silver", "gold"]

# Criar os buckets se não existirem
for bucket in buckets:
    if not client.bucket_exists(bucket):
        client.make_bucket(bucket)

# Fazer upload dos arquivos descompactados para o bucket raw no MinIO
try:
    for root, dirs, files in os.walk('../data/raw'):
        for file in files:
            file_path = os.path.join(root, file)
            client.fput_object(
                "raw", f"{file}", file_path
            )
    print("Upload para o MinIO concluído com sucesso!")
except S3Error as e:
    print(f"Erro ao fazer upload para o MinIO: {e}")

Configurações carregadas com sucesso!
Dataset URL: https://www.kaggle.com/datasets/utkarshx27/non-alcohol-fatty-liver-disease
Download do dataset concluído com sucesso!
Descompactação concluída com sucesso!
Upload para o MinIO concluído com sucesso!
